# Relaxationszeiten aus exponentiellen Verlaeufen

Dieses Notebook bestimmt Zeitkonstanten aus den zeitlichen Verlaeufen von

- der direkt bestimmten Stufenhoehe `delta epsilon'` (`Eps_real_Step_Height`),
- der aus dem Fit bestimmten dielektrischen Staerke `delta epsilon` (`de`),
- der Peakposition `omega_p`.

Gefittet wird pro Material, Temperatur, Modus und Observable ein exponentieller Verlauf

`y(t) = y_inf + A * exp(-t / tau)`

wobei `tau` die gesuchte Relaxationszeit ist.

Fuer `delta epsilon'` aus der Stufenhoehe wird je nach Modus geteilt: Bei Absorption laeuft ein Fit von `t = 0` bis zum Minimum und der zweite vom Minimum bis zum Ende. Bei Desorption laeuft ein Fit von `t = 0` bis zum ersten Wendepunkt und der zweite vom ersten bis zum zweiten Wendepunkt. Ausnahmen: Fuer 50 C Des wird `delta epsilon'` aus der Stufenhoehe fest von 0 bis 75 s und von 75 bis 425 s gefittet; fuer 60 C Des fest von 0 bis 200 s und von 200 bis 325 s. Fuer `delta epsilon` aus dem Fit wird normalerweise ein Fit ueber den ausgewaehlten Zeitbereich verwendet; fuer 70 C Abs wird `delta epsilon` aus dem Fit bei 1200 s in zwei Fits geteilt, fuer 50 C Des wird nur bis 3000 s gefittet, fuer 60 C Des bei 900 s geteilt, fuer 70 C Des bei 350 s geteilt. `omega_p` bleibt im Datensatz verfuegbar, ist aber aktuell nicht in den Relaxations-Fits enthalten.

In [ ]:
from pathlib import Path
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
from scipy.signal import savgol_filter

plt.style.use("default")
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == "code" else Path.cwd()
RESULTS_DIR = PROJECT_ROOT / "results"
PLOT_DIR = RESULTS_DIR / "relaxation_time_plots"

PROJECT_ROOT, RESULTS_DIR

## Einstellungen

`FIT_AFTER_SWITCH_ONLY = True` verwendet nur Messpunkte ab dem Schaltpunkt (`Time_Relative_s >= 0`). Wenn du den gesamten Verlauf fitten moechtest, setze den Wert auf `False`.

`FIT_TEMPERATURES_C = [50, 60, 70]` schliesst 80 C und 90 C aus den Relaxations-Fits aus. Die Daten bleiben aber fuer die Darstellung verfuegbar.

`FIT_OBSERVABLES` legt fest, welche Observables gefittet werden. `omega_p` wird aktuell nicht gefittet.

In [ ]:
USE_PILOT_FITS = False
FIT_AFTER_SWITCH_ONLY = True
MIN_TIME_S = 0.0
MIN_POINTS_PER_FIT = 6
FIT_TEMPERATURES_C = [50, 60, 70]
FIT_OBSERVABLES = ["delta_eps_real_step_height", "delta_eps_fit_de"]

# Optional einschraenken, z.B. TEMPERATURES_C = [50, 60, 70]
MATERIALS = None
TEMPERATURES_C = None
MODES = None

FIT_FILE_PATTERN = "fit_parameters_pilot_*.csv" if USE_PILOT_FITS else "fit_parameters_*.csv ohne pilot"
FIT_FILE_PATTERN

In [ ]:
def parse_temperature_c(value):
    match = re.search(r"(\d+(?:\.\d+)?)", str(value))
    return float(match.group(1)) if match else np.nan


def parse_fit_filename(path):
    match = re.match(
        r"fit_parameters(?:_pilot)?_(?P<material>.+?)_(?P<temperature>\d+)C_(?P<mode>Abs|Des)\.csv$",
        path.name,
    )
    if not match:
        raise ValueError(f"Dateiname passt nicht zum erwarteten Muster: {path.name}")
    return {
        "Material": match.group("material"),
        "Temperature_C": float(match.group("temperature")),
        "Mode": match.group("mode"),
    }


def load_direct_step_heights():
    all_series = RESULTS_DIR / "step_height_eps_real_all_series.csv"
    if all_series.exists():
        df = pd.read_csv(all_series)
    else:
        files = sorted(RESULTS_DIR.glob("step_height_eps_real_*C_*.csv"))
        df = pd.concat((pd.read_csv(path) for path in files), ignore_index=True)

    df = df.copy()
    df["Temperature_C"] = df["Temperature"].map(parse_temperature_c)
    return df


def load_fit_parameters():
    if USE_PILOT_FITS:
        files = sorted(RESULTS_DIR.glob("fit_parameters_pilot_*.csv"))
    else:
        files = sorted(
            path for path in RESULTS_DIR.glob("fit_parameters_*.csv")
            if not path.name.startswith("fit_parameters_pilot_")
        )
    if not files:
        raise FileNotFoundError(f"Keine Fitparameter-Dateien gefunden: {FIT_FILE_PATTERN}")

    frames = []
    for path in files:
        df = pd.read_csv(path)
        for key, value in parse_fit_filename(path).items():
            df[key] = value
        df["Source_File"] = path.name
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


direct_df = load_direct_step_heights()
fit_df = load_fit_parameters()

direct_df.head(), fit_df.head()

In [ ]:
def apply_selection(df):
    selected = df.copy()
    if MATERIALS is not None:
        selected = selected[selected["Material"].isin(MATERIALS)]
    if TEMPERATURES_C is not None:
        selected = selected[selected["Temperature_C"].isin(TEMPERATURES_C)]
    if MODES is not None:
        selected = selected[selected["Mode"].isin(MODES)]
    return selected


def to_observable_frame(df, value_column, observable, source):
    required = ["Material", "Temperature_C", "Mode", "Time_Relative_s", value_column]
    missing = [column for column in required if column not in df.columns]
    if missing:
        raise KeyError(f"Fehlende Spalten fuer {observable}: {missing}")

    out = df[required].copy()
    out = out.rename(columns={value_column: "value"})
    out["observable"] = observable
    out["source"] = source
    return out


direct_selected = apply_selection(direct_df)
fit_selected = apply_selection(fit_df)

if "Accepted" in fit_selected.columns:
    fit_selected = fit_selected[fit_selected["Accepted"].fillna(False).astype(bool)]
if "Fit_Success" in fit_selected.columns:
    fit_selected = fit_selected[fit_selected["Fit_Success"].fillna(False).astype(bool)]

observables = [
    to_observable_frame(direct_selected, "Eps_real_Step_Height", "delta_eps_real_step_height", "step_height"),
    to_observable_frame(fit_selected, "de", "delta_eps_fit_de", "fit"),
    to_observable_frame(fit_selected, "omega_p", "omega_p", "fit"),
]

data_long = pd.concat(observables, ignore_index=True)
data_long = data_long.replace([np.inf, -np.inf], np.nan).dropna(subset=["Time_Relative_s", "value"])
data_long = data_long.sort_values(["Material", "Temperature_C", "Mode", "observable", "Time_Relative_s"])

data_long.groupby(["Material", "Temperature_C", "Mode", "observable"]).size().rename("n_points").reset_index()

## Delta epsilon darstellen

In diesem Block koennen `delta epsilon'` aus der Stufenhoehe und `delta epsilon` aus dem Fit dargestellt werden. Von den drei Parametern `Variable`, `Modus` und `Temperatur` muessen immer genau zwei festgehalten werden. Der dritte Parameter wird dann im Plot variiert.

In [ ]:
DELTA_OBSERVABLES = {
    "delta_eps_real_step_height": "Delta epsilon' aus Stufenhoehe",
    "delta_eps_fit_de": "Delta epsilon aus Fit (de)",
}


def is_multi_selection(value):
    return isinstance(value, (list, tuple, set, np.ndarray, pd.Series)) and not isinstance(value, str)


def selection_values(value, *, as_float=False):
    if value is None:
        return None
    values = list(value) if is_multi_selection(value) else [value]
    if as_float:
        return [float(item) for item in values]
    return values


def selection_is_varied(value):
    return value is None or (is_multi_selection(value) and len(value) != 1)


def format_selection(value, unit=""):
    if value is None:
        return "alle"
    values = list(value) if is_multi_selection(value) else [value]
    formatted = []
    for item in values:
        if isinstance(item, (int, float, np.integer, np.floating)):
            formatted.append(f"{float(item):g}{unit}")
        else:
            formatted.append(f"{item}{unit}")
    return ", ".join(formatted)


def normalize_axis_scale(scale, *, log_flag=False):
    if log_flag:
        return "log"
    if scale is None:
        return "linear"
    scale = str(scale).lower()
    if scale not in {"linear", "log", "symlog"}:
        raise ValueError("Achsen-Skala muss 'linear', 'log' oder 'symlog' sein.")
    return scale


def plot_delta_epsilon(
    fixed_variable=None,
    fixed_mode=None,
    fixed_temperature_c=None,
    x_scale="linear",
    y_scale="linear",
    log_x=False,
    log_y=False,
    symlog_linthresh_x=1.0,
    symlog_linthresh_y=1e-3,
    xlim=None,
    ylim=None,
    figsize=(7.2, 4.4),
):
    x_scale = normalize_axis_scale(x_scale, log_flag=log_x)
    y_scale = normalize_axis_scale(y_scale, log_flag=log_y)
    fixed = {
        "Variable": fixed_variable,
        "Modus": fixed_mode,
        "Temperatur": fixed_temperature_c,
    }
    varied_parameters = [name for name, value in fixed.items() if selection_is_varied(value)]
    if len(varied_parameters) > 1:
        raise ValueError("Bitte hoechstens einen Parameter variieren: None fuer alle Werte oder Liste/Tupel fuer eine Auswahl.")

    plot_df = data_long[data_long["observable"].isin(DELTA_OBSERVABLES)].copy()
    variable_values = selection_values(fixed_variable)
    mode_values = selection_values(fixed_mode)
    temperature_values = selection_values(fixed_temperature_c, as_float=True)

    if variable_values is not None:
        plot_df = plot_df[plot_df["observable"].isin(variable_values)]
    if mode_values is not None:
        plot_df = plot_df[plot_df["Mode"].isin(mode_values)]
    if temperature_values is not None:
        plot_df = plot_df[plot_df["Temperature_C"].isin(temperature_values)]

    if plot_df.empty:
        raise ValueError("Keine Daten fuer diese Auswahl gefunden.")
    if x_scale == "log":
        plot_df = plot_df[plot_df["Time_Relative_s"] > 0]
    if y_scale == "log":
        plot_df = plot_df[plot_df["value"] > 0]
    if plot_df.empty:
        raise ValueError("Keine positiven Datenpunkte fuer die gewaehlte logarithmische Darstellung gefunden.")

    if len(varied_parameters) == 0:
        vary_column = None
        vary_label = "keine Variation"
    elif selection_is_varied(fixed_variable):
        vary_column = "observable"
        vary_label = "Variable"
    elif selection_is_varied(fixed_mode):
        vary_column = "Mode"
        vary_label = "Modus"
    else:
        vary_column = "Temperature_C"
        vary_label = "Temperatur"

    fig, ax = plt.subplots(figsize=figsize)
    if vary_column is None:
        group_columns = ["Material"] if "Material" in plot_df.columns and plot_df["Material"].nunique() > 1 else []
    else:
        group_columns = [vary_column]
        if "Material" in plot_df.columns and plot_df["Material"].nunique() > 1:
            group_columns = ["Material", vary_column]

    grouped = [("Auswahl", plot_df)] if not group_columns else plot_df.groupby(group_columns)
    for key, group in grouped:
        group = group.sort_values("Time_Relative_s")
        key_parts = key if isinstance(key, tuple) else (key,)
        vary_value = key_parts[-1]
        if vary_column is None:
            label = "Auswahl"
        elif vary_column == "observable":
            label = DELTA_OBSERVABLES.get(vary_value, vary_value)
        elif vary_column == "Temperature_C":
            label = f"{float(vary_value):g} C"
        else:
            label = str(vary_value)

        if len(key_parts) > 1:
            label = " | ".join(str(part) for part in key_parts[:-1] + (label,))

        ax.plot(group["Time_Relative_s"], group["value"], marker=".", linewidth=1.2, markersize=2.5, label=label)

    title_parts = []
    for name, value in fixed.items():
        if name == "Temperatur":
            title_parts.append(f"{name}: {format_selection(value, ' C')}")
        else:
            title_parts.append(f"{name}: {format_selection(value)}")
    if vary_column is None:
        ax.set_title("Delta epsilon | " + ", ".join(title_parts))
    else:
        ax.set_title("Delta epsilon, variiert: " + vary_label + " | " + ", ".join(title_parts))
    ax.set_xlabel("relative Zeit / s")
    ax.set_ylabel("Delta epsilon")
    if x_scale == "log":
        ax.set_xscale("log")
    elif x_scale == "symlog":
        ax.set_xscale("symlog", linthresh=symlog_linthresh_x)
    if y_scale == "log":
        ax.set_yscale("log")
    elif y_scale == "symlog":
        ax.set_yscale("symlog", linthresh=symlog_linthresh_y)
    if xlim is not None:
        ax.set_xlim(xlim)
    if ylim is not None:
        ax.set_ylim(ylim)
    ax.legend(title=None if vary_column is None else vary_label)
    fig.tight_layout()
    return fig, ax


# Beispiele: zwei Parameter skalar festhalten, der dritte wird variiert.
# plot_delta_epsilon(fixed_variable="delta_eps_fit_de", fixed_mode="Abs")
# plot_delta_epsilon(fixed_variable="delta_eps_real_step_height", fixed_temperature_c=50)
# plot_delta_epsilon(fixed_mode="Abs", fixed_temperature_c=50)
# plot_delta_epsilon(fixed_variable="delta_eps_real_step_height", fixed_mode="Des", fixed_temperature_c=(50, 60, 70), x_scale="log")
# plot_delta_epsilon(fixed_variable="delta_eps_fit_de", fixed_mode="Abs", fixed_temperature_c=None, x_scale="log", y_scale="log")
# plot_delta_epsilon(fixed_variable="delta_eps_real_step_height", fixed_mode="Abs", fixed_temperature_c=None, x_scale="symlog")

In [ ]:
VARIABLE = "delta_eps_real_step_height"  # "delta_eps_real_step_height" oder "delta_eps_fit_de"
MODUS = "Abs"  # "Abs", "Des" oder None
TEMPERATUR_C = None  # z.B. 50, (50, 60, 70) oder None

X_SCALE = "linear"  # "linear", "log" oder "symlog"
Y_SCALE = "linear"  # "linear", "log" oder "symlog"
X_LIM = None  # z.B. (-100, 5000) oder None fuer automatisch
Y_LIM = None  # z.B. (2.0, 4.5) oder None fuer automatisch
FIGSIZE = (7.2, 4.4)  # Breite, Hoehe in inch

plot_delta_epsilon(
    fixed_variable=VARIABLE,
    fixed_mode=MODUS,
    fixed_temperature_c=TEMPERATUR_C,
    x_scale=X_SCALE,
    y_scale=Y_SCALE,
    xlim=X_LIM,
    ylim=Y_LIM,
    figsize=FIGSIZE,
);

In [ ]:
def exp_model(t, y_inf, amplitude, tau):
    return y_inf + amplitude * np.exp(-t / tau)


STEP_HEIGHT_OBSERVABLE = "delta_eps_real_step_height"
FIT_DE_OBSERVABLE = "delta_eps_fit_de"
DE_70_ABS_SPLIT_TIME_S = 1200.0
DE_70_DES_SPLIT_TIME_S = 350.0
DE_50_DES_END_TIME_S = 3000.0
DE_60_DES_SPLIT_TIME_S = 900.0
STEP_HEIGHT_50_DES_RANGES_S = [(0.0, 75.0), (75.0, 425.0)]
STEP_HEIGHT_60_DES_RANGES_S = [(0.0, 200.0), (200.0, 325.0)]


def prepare_fit_group(group):
    group = group.sort_values("Time_Relative_s").copy()
    if FIT_AFTER_SWITCH_ONLY:
        group = group[group["Time_Relative_s"] >= MIN_TIME_S]
    return group.dropna(subset=["Time_Relative_s", "value"])


def split_at_minimum(group):
    group = prepare_fit_group(group)
    if group.empty:
        return [("to_minimum", group, {}), ("from_minimum", group, {})]

    min_idx = group["value"].idxmin()
    min_time = float(group.loc[min_idx, "Time_Relative_s"])
    min_value = float(group.loc[min_idx, "value"])
    metadata = {
        "minimum_time_s": min_time,
        "minimum_value": min_value,
        "split_point_1_s": min_time,
        "split_point_1_value": min_value,
        "split_point_1_type": "minimum",
        "split_point_2_s": np.nan,
        "split_point_2_value": np.nan,
        "split_point_2_type": "",
    }

    return [
        ("to_minimum", group[group["Time_Relative_s"] <= min_time], metadata),
        ("from_minimum", group[group["Time_Relative_s"] >= min_time], metadata),
    ]


def smooth_for_inflection(y):
    n = len(y)
    if n < 7:
        return y
    window = min(301, max(7, int(n * 0.05)))
    if window % 2 == 0:
        window += 1
    if window >= n:
        window = n - 1 if n % 2 == 0 else n
    if window < 7:
        return y
    return savgol_filter(y, window_length=window, polyorder=3)


def find_inflection_times(group):
    group = prepare_fit_group(group)
    if len(group) < MIN_POINTS_PER_FIT:
        return [], group

    t = group["Time_Relative_s"].to_numpy(dtype=float)
    y = group["value"].to_numpy(dtype=float)
    y_smooth = smooth_for_inflection(y)
    first_derivative = np.gradient(y_smooth, t)
    second_derivative = np.gradient(first_derivative, t)

    signs = np.sign(second_derivative)
    nonzero = signs != 0
    if nonzero.any():
        signs = pd.Series(signs).replace(0, np.nan).ffill().bfill().to_numpy()

    change_indices = np.where(np.diff(signs) != 0)[0] + 1
    margin = max(2, int(0.03 * len(group)))
    change_indices = [idx for idx in change_indices if margin <= idx < len(group) - margin]

    selected = []
    min_distance = max(2, int(0.08 * len(group)))
    for idx in change_indices:
        if not selected or idx - selected[-1] >= min_distance:
            selected.append(idx)
        if len(selected) == 2:
            break

    return [(float(t[idx]), float(y[idx])) for idx in selected], group


def split_at_des_inflections(group):
    inflections, group = find_inflection_times(group)
    if len(inflections) < 2:
        metadata = {
            "minimum_time_s": np.nan,
            "minimum_value": np.nan,
            "split_point_1_s": np.nan,
            "split_point_1_value": np.nan,
            "split_point_1_type": "inflection",
            "split_point_2_s": np.nan,
            "split_point_2_value": np.nan,
            "split_point_2_type": "inflection",
        }
        return [("to_first_inflection", group.iloc[0:0], metadata), ("first_to_second_inflection", group.iloc[0:0], metadata)]

    (first_time, first_value), (second_time, second_value) = inflections[:2]
    metadata = {
        "minimum_time_s": np.nan,
        "minimum_value": np.nan,
        "split_point_1_s": first_time,
        "split_point_1_value": first_value,
        "split_point_1_type": "inflection",
        "split_point_2_s": second_time,
        "split_point_2_value": second_value,
        "split_point_2_type": "inflection",
    }
    return [
        ("to_first_inflection", group[group["Time_Relative_s"] <= first_time], metadata),
        ("first_to_second_inflection", group[(group["Time_Relative_s"] >= first_time) & (group["Time_Relative_s"] <= second_time)], metadata),
    ]


def split_step_height_group(group, mode):
    if mode == "Des":
        return split_at_des_inflections(group)
    return split_at_minimum(group)


def split_at_time(group, split_time_s):
    group = prepare_fit_group(group)
    if group.empty:
        return [("to_1200s", group, {}), ("from_1200s", group, {})]

    nearest_idx = (group["Time_Relative_s"] - split_time_s).abs().idxmin()
    split_value = float(group.loc[nearest_idx, "value"])
    metadata = {
        "minimum_time_s": np.nan,
        "minimum_value": np.nan,
        "split_point_1_s": float(split_time_s),
        "split_point_1_value": split_value,
        "split_point_1_type": "manual_time",
        "split_point_2_s": np.nan,
        "split_point_2_value": np.nan,
        "split_point_2_type": "",
    }
    label_time = f"{split_time_s:g}s"
    return [
        (f"to_{label_time}", group[group["Time_Relative_s"] <= split_time_s], metadata),
        (f"from_{label_time}", group[group["Time_Relative_s"] >= split_time_s], metadata),
    ]


def fit_until_time(group, end_time_s):
    group = prepare_fit_group(group)
    if group.empty:
        return [(f"to_{end_time_s:g}s", group, {})]

    nearest_idx = (group["Time_Relative_s"] - end_time_s).abs().idxmin()
    end_value = float(group.loc[nearest_idx, "value"])
    metadata = {
        "minimum_time_s": np.nan,
        "minimum_value": np.nan,
        "split_point_1_s": float(end_time_s),
        "split_point_1_value": end_value,
        "split_point_1_type": "manual_time",
        "split_point_2_s": np.nan,
        "split_point_2_value": np.nan,
        "split_point_2_type": "",
    }
    return [(f"to_{end_time_s:g}s", group[group["Time_Relative_s"] <= end_time_s], metadata)]


def split_into_time_ranges(group, ranges_s):
    group = prepare_fit_group(group)
    if group.empty:
        return [(f"{start:g}s_to_{end:g}s", group, {}) for start, end in ranges_s]

    split_times = sorted({float(time) for time_range in ranges_s for time in time_range})
    metadata = {
        "minimum_time_s": np.nan,
        "minimum_value": np.nan,
        "split_point_1_s": split_times[1] if len(split_times) > 1 else np.nan,
        "split_point_1_value": np.nan,
        "split_point_1_type": "manual_time",
        "split_point_2_s": split_times[2] if len(split_times) > 2 else np.nan,
        "split_point_2_value": np.nan,
        "split_point_2_type": "manual_time" if len(split_times) > 2 else "",
    }
    for point_key, value_key in [("split_point_1_s", "split_point_1_value"), ("split_point_2_s", "split_point_2_value")]:
        split_time = metadata[point_key]
        if pd.notna(split_time):
            nearest_idx = (group["Time_Relative_s"] - split_time).abs().idxmin()
            metadata[value_key] = float(group.loc[nearest_idx, "value"])

    segments = []
    for start_time, end_time in ranges_s:
        segment = group[(group["Time_Relative_s"] >= start_time) & (group["Time_Relative_s"] <= end_time)]
        segments.append((f"{start_time:g}s_to_{end_time:g}s", segment, metadata))
    return segments


def should_split_de_70_abs(row):
    return (
        row["observable"] == FIT_DE_OBSERVABLE
        and row["Mode"] == "Abs"
        and float(row["Temperature_C"]) == 70.0
    )


def should_fit_de_50_des_to_3000(row):
    return (
        row["observable"] == FIT_DE_OBSERVABLE
        and row["Mode"] == "Des"
        and float(row["Temperature_C"]) == 50.0
    )


def should_split_de_60_des(row):
    return (
        row["observable"] == FIT_DE_OBSERVABLE
        and row["Mode"] == "Des"
        and float(row["Temperature_C"]) == 60.0
    )


def should_split_de_70_des(row):
    return (
        row["observable"] == FIT_DE_OBSERVABLE
        and row["Mode"] == "Des"
        and float(row["Temperature_C"]) == 70.0
    )


def should_split_step_height_50_des(row):
    return (
        row["observable"] == STEP_HEIGHT_OBSERVABLE
        and row["Mode"] == "Des"
        and float(row["Temperature_C"]) == 50.0
    )


def should_split_step_height_60_des(row):
    return (
        row["observable"] == STEP_HEIGHT_OBSERVABLE
        and row["Mode"] == "Des"
        and float(row["Temperature_C"]) == 60.0
    )


def fit_exponential(group):
    group = prepare_fit_group(group)
    if len(group) < MIN_POINTS_PER_FIT:
        return None, group, "too_few_points"

    t_raw = group["Time_Relative_s"].to_numpy(dtype=float)
    y = group["value"].to_numpy(dtype=float)
    t = t_raw - np.nanmin(t_raw)

    if np.nanmax(t) <= 0 or np.nanstd(y) == 0:
        return None, group, "not_enough_variation"

    tail_n = max(3, len(y) // 5)
    y_inf0 = float(np.nanmedian(y[-tail_n:]))
    amplitude0 = float(y[0] - y_inf0)
    if amplitude0 == 0:
        amplitude0 = float(np.nanmax(y) - np.nanmin(y))
    tau0 = max(float(np.nanmax(t) / 3), 1e-9)

    y_span = max(float(np.nanmax(y) - np.nanmin(y)), 1e-12)
    lower = [float(np.nanmin(y) - 5 * y_span), -10 * y_span, 1e-9]
    upper = [float(np.nanmax(y) + 5 * y_span), 10 * y_span, max(float(np.nanmax(t) * 100), 1.0)]

    try:
        popt, pcov = curve_fit(
            exp_model,
            t,
            y,
            p0=[y_inf0, amplitude0, tau0],
            bounds=(lower, upper),
            maxfev=20000,
        )
    except Exception as exc:
        return None, group, f"fit_failed: {exc}"

    y_fit = exp_model(t, *popt)
    residuals = y - y_fit
    ss_res = float(np.sum(residuals**2))
    ss_tot = float(np.sum((y - np.mean(y))**2))
    r2 = np.nan if ss_tot == 0 else 1 - ss_res / ss_tot
    rmse = float(np.sqrt(np.mean(residuals**2)))

    perr = np.full(3, np.nan)
    if pcov is not None and np.all(np.isfinite(pcov)):
        perr = np.sqrt(np.diag(pcov))

    result = {
        "n_points": len(group),
        "time_start_s": float(np.nanmin(t_raw)),
        "time_end_s": float(np.nanmax(t_raw)),
        "y_inf": float(popt[0]),
        "amplitude": float(popt[1]),
        "tau_s": float(popt[2]),
        "tau_err_s": float(perr[2]) if np.isfinite(perr[2]) else np.nan,
        "rmse": rmse,
        "r2": float(r2) if np.isfinite(r2) else np.nan,
        "fit_status": "ok",
    }
    return result, group.assign(t_fit_s=t, y_fit=y_fit), "ok"


fit_rows = []
fit_curves = []
group_columns = ["Material", "Temperature_C", "Mode", "observable"]
fit_data_long = data_long.copy()
if FIT_TEMPERATURES_C is not None:
    fit_data_long = fit_data_long[fit_data_long["Temperature_C"].isin(FIT_TEMPERATURES_C)]
if FIT_OBSERVABLES is not None:
    fit_data_long = fit_data_long[fit_data_long["observable"].isin(FIT_OBSERVABLES)]

for group_key, group in fit_data_long.groupby(group_columns):
    base_row = dict(zip(group_columns, group_key))
    if should_split_step_height_50_des(base_row):
        fit_groups = split_into_time_ranges(group, STEP_HEIGHT_50_DES_RANGES_S)
    elif should_split_step_height_60_des(base_row):
        fit_groups = split_into_time_ranges(group, STEP_HEIGHT_60_DES_RANGES_S)
    elif base_row["observable"] == STEP_HEIGHT_OBSERVABLE:
        fit_groups = split_step_height_group(group, base_row["Mode"])
    elif should_fit_de_50_des_to_3000(base_row):
        fit_groups = fit_until_time(group, DE_50_DES_END_TIME_S)
    elif should_split_de_60_des(base_row):
        fit_groups = split_at_time(group, DE_60_DES_SPLIT_TIME_S)
    elif should_split_de_70_des(base_row):
        fit_groups = split_at_time(group, DE_70_DES_SPLIT_TIME_S)
    elif should_split_de_70_abs(base_row):
        fit_groups = split_at_time(group, DE_70_ABS_SPLIT_TIME_S)
    else:
        fit_groups = [("full", group, {})]

    for fit_segment, segment_group, segment_metadata in fit_groups:
        result, curve, status = fit_exponential(segment_group)
        row = base_row.copy()
        row.update({
            "fit_segment": fit_segment,
        })
        row.update(segment_metadata)
        if result is None:
            row.update({"fit_status": status, "n_points": len(curve), "tau_s": np.nan, "tau_err_s": np.nan, "r2": np.nan, "rmse": np.nan})
        else:
            row.update(result)
            fit_curves.append(curve.assign(**row))
        fit_rows.append(row)

relaxation_results = pd.DataFrame(fit_rows).sort_values(group_columns + ["fit_segment"])
relaxation_results

In [ ]:
def safe_name(value):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value)).strip("_")


def plot_group(group, result_row, save=False):
    material = result_row["Material"]
    temp = result_row["Temperature_C"]
    mode = result_row["Mode"]
    observable = result_row["observable"]
    fit_segment = result_row.get("fit_segment", "full")

    group = group.sort_values("Time_Relative_s").copy()
    if "time_start_s" in result_row and pd.notna(result_row["time_start_s"]):
        group = group[group["Time_Relative_s"] >= result_row["time_start_s"]]
    if "time_end_s" in result_row and pd.notna(result_row["time_end_s"]):
        group = group[group["Time_Relative_s"] <= result_row["time_end_s"]]

    fig, ax = plt.subplots(figsize=(6.4, 4.0))
    ax.scatter(group["Time_Relative_s"], group["value"], s=18, label="Daten")
    split_points = [
        (result_row.get("split_point_1_s", np.nan), result_row.get("split_point_1_type", "Split 1")),
        (result_row.get("split_point_2_s", np.nan), result_row.get("split_point_2_type", "Split 2")),
    ]
    for split_time, split_type in split_points:
        if pd.notna(split_time):
            if split_type == "minimum":
                label = "Minimum"
            elif split_type == "inflection":
                label = "Wendepunkt"
            elif split_type == "manual_time":
                label = f"Split bei {split_time:g} s"
            else:
                label = "Split"
            ax.axvline(split_time, color="tab:gray", linestyle="--", linewidth=1.0, label=label)

    if result_row.get("fit_status") == "ok":
        t_raw = group["Time_Relative_s"].to_numpy(dtype=float)
        t_plot_raw = np.linspace(np.nanmin(t_raw), np.nanmax(t_raw), 300)
        t_plot = t_plot_raw - np.nanmin(t_raw)
        y_plot = exp_model(t_plot, result_row["y_inf"], result_row["amplitude"], result_row["tau_s"])
        ax.plot(t_plot_raw, y_plot, color="tab:red", label=f"Fit {fit_segment}: tau = {result_row['tau_s']:.3g} s")

    title_segment = "" if fit_segment == "full" else f" ({fit_segment})"
    ax.set_title(f"{material}, {temp:g} C, {mode}: {observable}{title_segment}")
    ax.set_xlabel("relative Zeit / s")
    ax.set_ylabel(observable)
    ax.legend()
    fig.tight_layout()

    if save:
        PLOT_DIR.mkdir(parents=True, exist_ok=True)
        filename = f"relaxation_{safe_name(material)}_{temp:g}C_{mode}_{safe_name(observable)}_{safe_name(fit_segment)}.png"
        fig.savefig(PLOT_DIR / filename, bbox_inches="tight")
    return fig, ax


with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=RuntimeWarning)
    for _, result_row in relaxation_results.iterrows():
        if result_row["fit_status"] != "ok":
            continue
        mask = (
            (data_long["Material"] == result_row["Material"])
            & (data_long["Temperature_C"] == result_row["Temperature_C"])
            & (data_long["Mode"] == result_row["Mode"])
            & (data_long["observable"] == result_row["observable"])
        )
        group = data_long.loc[mask].sort_values("Time_Relative_s")
        if FIT_AFTER_SWITCH_ONLY:
            group = group[group["Time_Relative_s"] >= MIN_TIME_S]
        plot_group(group, result_row, save=False)
        plt.show()

print("Es wurde nichts gespeichert. Zum Speichern die letzte Zelle verwenden.")

In [ ]:
summary = relaxation_results.pivot_table(
    index=["Material", "Temperature_C", "Mode"],
    columns=["observable", "fit_segment"],
    values="tau_s",
    aggfunc="first",
)
summary

## Optional speichern

Erst wenn die Ergebnisse plausibel aussehen, `SPEICHERN = True` setzen und diese Zelle ausfuehren.

In [ ]:
SPEICHERN = False

if SPEICHERN:
    output_csv = RESULTS_DIR / "relaxation_times_exponential_fits.csv"
    relaxation_results.to_csv(output_csv, index=False)

    PLOT_DIR.mkdir(parents=True, exist_ok=True)
    saved_plots = 0
    for _, result_row in relaxation_results.iterrows():
        if result_row["fit_status"] != "ok":
            continue
        mask = (
            (data_long["Material"] == result_row["Material"])
            & (data_long["Temperature_C"] == result_row["Temperature_C"])
            & (data_long["Mode"] == result_row["Mode"])
            & (data_long["observable"] == result_row["observable"])
        )
        group = data_long.loc[mask].sort_values("Time_Relative_s")
        if FIT_AFTER_SWITCH_ONLY:
            group = group[group["Time_Relative_s"] >= MIN_TIME_S]
        fig, _ = plot_group(group, result_row, save=True)
        plt.close(fig)
        saved_plots += 1

    print(f"Gespeichert: {output_csv}")
    print(f"Gespeicherte Plots: {saved_plots} in {PLOT_DIR}")
else:
    print("Nichts gespeichert. Setze SPEICHERN = True, wenn du die Ergebnisse exportieren willst.")